In [ ]:
import itertools
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.surface_code.patch import PauliBasis
from library.surface_code.teleportation import TeleportationSurgery
from library.qubit_array import QubitArray

In [ ]:
circuit = Circuitry()
array = QubitArray(circuit, dimensions = (9, 5))
basis = PauliBasis.Z

surgery = TeleportationSurgery(array, distance = 3, anchor = (1, 1))
surgery.append_movement(circuit, prepare = basis, measure = basis)

print(array.measurements_index)

for qubit, records in array.measurements_qubit.items():
    if len(records) > 1:
        patches = { key for key, _ in itertools.groupby(records, key=lambda mr: mr[0]) }

        print(f"Handling {patches} : {records}")
        if len(patches) == 1:
            # Purely in SOURCE or TARGET patches
            stabilizer = records[0].split(":")[-1]
            if stabilizer.startswith(basis.name):
                array.annotate_detector(circuit, records[0])

            for rounds in itertools.pairwise(records):
                array.annotate_detector(circuit, *rounds)
        elif len(patches) == 2: # Crossing SOURCE&MERGER or MERGER&TARGET patches
            stabilizer = records[0].split(":")[-1]
            if stabilizer.startswith(basis.name):
                array.annotate_detector(circuit, records[0])

            for sub_records in [ records[:3], records[3:] ]:
                for rounds in itertools.pairwise(sub_records):
                    array.annotate_detector(circuit, *rounds)

array.annotate_detector(circuit, 'S:M1:R2:Z0', 'S:M1:R2:D0', 'S:M1:R2:D1', 'S:M1:R2:D3', 'S:M1:R2:D4')
array.annotate_detector(circuit, 'S:M1:R2:Z2', 'S:M1:R2:D1', 'S:M1:R2:D2')
array.annotate_detector(circuit, 'S:M1:R2:Z3', 'S:M1:R2:D4', 'S:M1:R2:D5', 'S:M1:R2:D7', 'S:M1:R2:D8')

array.annotate_detector(circuit, 'S:M0:R2:Z1', 'M:M1:R0:Z0')

array.annotate_detector(circuit, 'M:M1:R2:Z0', 'S:M1:R2:D6', 'S:M1:R2:D7', 'M:M1:R2:D3', 'M:M1:R2:D4')
array.annotate_detector(circuit, 'M:M1:R2:Z3', 'M:M1:R2:D4', 'M:M1:R2:D5', 'T:M2:R0:Z2')

array.annotate_detector(circuit, 'T:M2:R2:Z0', 'T:M2:R2:D0', 'T:M2:R2:D1', 'T:M2:R2:D3', 'T:M2:R2:D4')
array.annotate_detector(circuit, 'T:M2:R2:Z1', 'T:M2:R2:D6', 'T:M2:R2:D7')
array.annotate_detector(circuit, 'T:M2:R2:Z2', 'T:M2:R2:D1', 'T:M2:R2:D2')
array.annotate_detector(circuit, 'T:M2:R2:Z3', 'T:M2:R2:D4', 'T:M2:R2:D5', 'T:M2:R2:D7', 'T:M2:R2:D8')

surgery.annotate_observable(circuit, 0, 'S:M1:R2:D1', 'S:M1:R2:D4', 'S:M1:R2:D7', 'M:M1:R2:D4', 'T:M2:R2:D1', 'T:M2:R2:D4', 'T:M2:R2:D7')

missing = circuit.as_stim.missing_detectors()
print(f"Missing detectors : {len(missing)}")
for detector in missing:
    print(f"> Detector: {detector}")

display(Markdown(f"[Open in Crumble]({circuit.as_stim.to_crumble_url()})"))

In [ ]:
for label, _ in array.measurements_index.items():
    print(f"Label {label} : {array.retrieve_measurement(label)}")

In [ ]:
circuit.to_file("../generated/logical-teleportation.stim")